# WM-811K Term Project — Step 4

## 시너지 분석: 클래스 불균형 처리(Focal) × Knowledge Distillation

**2×2 매트릭스**:

|  | KD 없음 | KD 적용 |
|---|---|---|
| **Focal 없음** | ① student_none (Step 2) | ③ KD만 (Step 3) |
| **Focal 적용** | ② focal만 (Step 2) | ④ **Focal + KD (이 노트북)** |

①②③은 이미 결과가 있으므로, 이 노트북에서는 **④ (Focal + KD)** 만 새로 학습합니다.

**시너지 판정**: ④의 향상폭이 ②, ③ 각각의 향상폭의 합보다 크면 시너지 존재.
즉, (④ - ①) > (② - ①) + (③ - ①) 이면 상승 시너지.

**최적 설정** (Step 3에서 결정): T=5, alpha=0.5, ResNet-18 Teacher, StudentCNN
**평가**: macro-F1 + 소수 클래스(Near-full/Donut/Random/Scratch) 평균 F1

## 1. 환경 설정

In [1]:
!nvidia-smi

Sun May 31 14:32:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT_DIR = '/content/drive/MyDrive/wm811k_term'
DATA_PATH = os.path.join(PROJECT_DIR, 'data', 'wm811k_64.npz')
CKPT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')
TEACHER_CKPT = os.path.join(CKPT_DIR, 'teacher_none_aug0.pt')
assert os.path.exists(DATA_PATH) and os.path.exists(TEACHER_CKPT)
print('데이터/Teacher 확인 완료')

Mounted at /content/drive
데이터/Teacher 확인 완료


In [3]:
import random, numpy as np, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as tvm
from sklearn.metrics import f1_score
import json

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
def get_device():
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_seed(42); device=get_device(); print('Device:', device)

Device: cuda


In [4]:
d = np.load(DATA_PATH, allow_pickle=True)
X_train,y_train = d['X_train'],d['y_train']
X_test,y_test = d['X_test'],d['y_test']
CLASSES=list(d['classes']); NUM_CLASSES=len(CLASSES)
MINORITY=['Near-full','Donut','Random','Scratch']; MIN_IDX=[CLASSES.index(c) for c in MINORITY]
print('소수 클래스:', MINORITY)

소수 클래스: ['Near-full', 'Donut', 'Random', 'Scratch']


## 2. 데이터 / 모델 / 손실 / 평가 (이전 Step과 동일)

In [5]:
class WaferDataset(Dataset):
    def __init__(self, X, y, train=False, augment=False):
        self.X,self.y,self.train,self.augment=X,y,train,augment
    def __len__(self): return len(self.X)
    def __getitem__(self,i):
        img=self.X[i].astype(np.float32)/2.0
        if self.train and self.augment:
            if random.random()<0.5: img=np.fliplr(img).copy()
            if random.random()<0.5: img=np.flipud(img).copy()
        return torch.from_numpy(img).unsqueeze(0), int(self.y[i])

def make_loaders(batch_size=128, augment=False):
    tr=DataLoader(WaferDataset(X_train,y_train,True,augment),batch_size=batch_size,
                  shuffle=True,num_workers=8,pin_memory=True)
    te=DataLoader(WaferDataset(X_test,y_test,False,False),batch_size=256,
                  shuffle=False,num_workers=8,pin_memory=True)
    return tr,te

def get_teacher_r18(n=NUM_CLASSES):
    m=tvm.resnet18(weights=None)
    m.conv1=nn.Conv2d(1,64,3,stride=1,padding=1,bias=False); m.maxpool=nn.Identity()
    m.fc=nn.Linear(m.fc.in_features,n); return m

class StudentCNN(nn.Module):
    def __init__(self,n=NUM_CLASSES):
        super().__init__()
        def b(ci,co): return nn.Sequential(nn.Conv2d(ci,co,3,padding=1,bias=False),
            nn.BatchNorm2d(co),nn.ReLU(inplace=True),nn.MaxPool2d(2))
        self.features=nn.Sequential(b(1,32),b(32,64),b(64,128),b(128,256))
        self.gap=nn.AdaptiveAvgPool2d(1); self.classifier=nn.Linear(256,n)
    def forward(self,x):
        x=self.features(x); x=self.gap(x).flatten(1); return self.classifier(x)
def get_student_cnn(n=NUM_CLASSES): return StudentCNN(n)

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None):
        super().__init__(); self.gamma=gamma; self.alpha=alpha
    def forward(self, logits, target):
        logp=F.log_softmax(logits,dim=1); p=torch.exp(logp)
        logp_t=logp.gather(1,target.unsqueeze(1)).squeeze(1)
        p_t=p.gather(1,target.unsqueeze(1)).squeeze(1)
        loss=-((1-p_t)**self.gamma)*logp_t
        if self.alpha is not None: loss=self.alpha.to(logits.device)[target]*loss
        return loss.mean()

def kd_focal_loss(s_logits, t_logits, targets, T, alpha, focal):
    soft=F.kl_div(F.log_softmax(s_logits/T,dim=1),F.softmax(t_logits/T,dim=1),
                  reduction='batchmean')*(T*T)
    hard=focal(s_logits, targets)
    return alpha*soft + (1-alpha)*hard

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); preds=[]; trues=[]
    for x,y in loader:
        x=x.to(device,non_blocking=True)
        preds.append(model(x).argmax(1).cpu().numpy()); trues.append(y.numpy())
    pred,true=np.concatenate(preds),np.concatenate(trues)
    acc=(pred==true).mean()
    macro=f1_score(true,pred,average='macro',zero_division=0)
    per=f1_score(true,pred,average=None,labels=list(range(NUM_CLASSES)),zero_division=0)
    return {'accuracy':float(acc),'macro_f1':float(macro),
            'minority_f1':float(np.array(per)[MIN_IDX].mean()),'per_class_f1':per.tolist()}
print('정의 완료')

정의 완료


## 3. ④ Focal + KD 학습
hard label 부분을 CE 대신 **Focal Loss**로 대체한 KD.
soft label(Teacher) + focal hard label을 alpha=0.5로 결합.

In [6]:
def run_focal_kd(T=5.0, alpha=0.5, gamma=2.0, epochs=30, batch=128, lr=0.05, tag='kd_focal'):
    set_seed(42)
    teacher = get_teacher_r18().to(device)
    teacher.load_state_dict(torch.load(TEACHER_CKPT, map_location=device)); teacher.eval()
    tr_loader, te_loader = make_loaders(batch)
    student = get_student_cnn().to(device)
    focal = FocalLoss(gamma=gamma)
    optimizer = torch.optim.SGD(student.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler('cuda', enabled=True)
    ckpt = os.path.join(CKPT_DIR, f'{tag}.pt')
    print(f'>>> Focal+KD 시작 (T={T}, alpha={alpha}, gamma={gamma})')
    history, best = [], 0.0
    for ep in range(1, epochs+1):
        student.train(); ls=tot=0
        for x,y in tr_loader:
            x,y=x.to(device,non_blocking=True),y.to(device,non_blocking=True)
            with torch.no_grad(): t_logits=teacher(x)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda', enabled=True):
                s_logits=student(x)
                loss=kd_focal_loss(s_logits,t_logits,y,T,alpha,focal)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            ls+=loss.item()*x.size(0); tot+=x.size(0)
        scheduler.step()
        mm=evaluate(student, te_loader)
        history.append({'epoch':ep,'loss':ls/tot,**{k:mm[k] for k in ['accuracy','macro_f1','minority_f1']}})
        if ep%5==0 or ep==1:
            print(f"  Epoch {ep:02d} | loss {ls/tot:.3f} | acc {mm['accuracy']:.4f} | macroF1 {mm['macro_f1']:.4f} | 소수 {mm['minority_f1']:.4f}")
        if mm['macro_f1']>best:
            best=mm['macro_f1']; torch.save(student.state_dict(), ckpt)
    student.load_state_dict(torch.load(ckpt, map_location=device))
    final=evaluate(student, te_loader)
    print(f"[{tag}] 최종: acc {final['accuracy']*100:.2f}% | macroF1 {final['macro_f1']:.4f} | 소수평균 {final['minority_f1']:.4f}")
    with open(os.path.join(RESULTS_DIR, f'{tag}.json'),'w') as f:
        json.dump({'T':T,'alpha':alpha,'gamma':gamma,'tag':tag,**final,'classes':CLASSES,'history':history}, f, indent=2)
    return final

focal_kd = run_focal_kd(T=5.0, alpha=0.5, gamma=2.0, tag='kd_focal_T5_a0.5')

>>> Focal+KD 시작 (T=5.0, alpha=0.5, gamma=2.0)
  Epoch 01 | loss 0.804 | acc 0.9477 | macroF1 0.6147 | 소수 0.4059
  Epoch 05 | loss 0.344 | acc 0.9410 | macroF1 0.7667 | 소수 0.6675
  Epoch 10 | loss 0.278 | acc 0.9668 | macroF1 0.7933 | 소수 0.7241
  Epoch 15 | loss 0.225 | acc 0.9686 | macroF1 0.8306 | 소수 0.7937
  Epoch 20 | loss 0.162 | acc 0.9695 | macroF1 0.7987 | 소수 0.7289
  Epoch 25 | loss 0.099 | acc 0.9738 | macroF1 0.8557 | 소수 0.8215
  Epoch 30 | loss 0.074 | acc 0.9749 | macroF1 0.8633 | 소수 0.8304
[kd_focal_T5_a0.5] 최종: acc 97.47% | macroF1 0.8649 | 소수평균 0.8366


## 4. 2×2 시너지 종합
①②③ 결과를 Drive의 JSON에서 불러와 ④와 합쳐 분석합니다.
- ① student_none (Step 2)
- ② student_focal (Step 2)
- ③ kd_r18_T5_a0.5 (Step 3)
- ④ kd_focal_T5_a0.5 (이 노트북)

In [9]:
def load_result(name):
    with open(os.path.join(RESULTS_DIR, f'{name}.json')) as f:
        return json.load(f)

# minority_f1이 없는 결과(Step 2)는 per_class_f1에서 계산
def get_min_f1(v):
    if 'minority_f1' in v:
        return v['minority_f1']
    return float(np.array(v['per_class_f1'])[MIN_IDX].mean())

cells_2x2 = {
    '① none (baseline)': load_result('student_none'),
    '② focal only':      load_result('student_focal'),
    '③ KD only':         load_result('kd_r18_T5_a0.5'),
    '④ focal + KD':      load_result('kd_focal_T5_a0.5'),
}

print(f"{'조건':<24}{'Macro-F1':<12}{'소수평균-F1':<12}")
print('-'*48)
for k,v in cells_2x2.items():
    print(f"{k:<24}{v['macro_f1']:<12.4f}{get_min_f1(v):<12.4f}")

# 시너지 (macro-F1)
b  = cells_2x2['① none (baseline)']['macro_f1']
f_ = cells_2x2['② focal only']['macro_f1']
k_ = cells_2x2['③ KD only']['macro_f1']
fk = cells_2x2['④ focal + KD']['macro_f1']
gain_focal = f_ - b
gain_kd    = k_ - b
gain_both  = fk - b
synergy    = gain_both - (gain_focal + gain_kd)
print('\n=== 시너지 분석 (Macro-F1) ===')
print(f"  focal 단독 향상 (②-①): {gain_focal*100:+.2f}%p")
print(f"  KD 단독 향상   (③-①): {gain_kd*100:+.2f}%p")
print(f"  결합 향상      (④-①): {gain_both*100:+.2f}%p")
print(f"  시너지 (결합 - 단독합): {synergy*100:+.2f}%p")
print(f"  → {'상승 시너지 존재' if synergy>0 else '시너지 없음(독립적/상쇄)'}")

# 시너지 (소수 클래스)
b2  = get_min_f1(cells_2x2['① none (baseline)'])
f2  = get_min_f1(cells_2x2['② focal only'])
k2  = get_min_f1(cells_2x2['③ KD only'])
fk2 = get_min_f1(cells_2x2['④ focal + KD'])
syn2 = (fk2-b2) - ((f2-b2)+(k2-b2))
print('\n=== 시너지 분석 (소수 클래스 평균 F1) ===')
print(f"  focal 단독 (②-①): {(f2-b2)*100:+.2f}%p")
print(f"  KD 단독   (③-①): {(k2-b2)*100:+.2f}%p")
print(f"  결합      (④-①): {(fk2-b2)*100:+.2f}%p")
print(f"  시너지: {syn2*100:+.2f}%p → {'상승 시너지' if syn2>0 else '독립적/상쇄'}")

조건                      Macro-F1    소수평균-F1     
------------------------------------------------
① none (baseline)       0.8606      0.8398      
② focal only            0.8569      0.8334      
③ KD only               0.8732      0.8498      
④ focal + KD            0.8649      0.8366      

=== 시너지 분석 (Macro-F1) ===
  focal 단독 향상 (②-①): -0.38%p
  KD 단독 향상   (③-①): +1.25%p
  결합 향상      (④-①): +0.43%p
  시너지 (결합 - 단독합): -0.44%p
  → 시너지 없음(독립적/상쇄)

=== 시너지 분석 (소수 클래스 평균 F1) ===
  focal 단독 (②-①): -0.64%p
  KD 단독   (③-①): +1.00%p
  결합      (④-①): -0.32%p
  시너지: -0.68%p → 독립적/상쇄


## 5. 클래스별 F1 비교 (4개 조건)

In [10]:
print(f"{'클래스':<12}" + ''.join(f'{c:<14}' for c in ['①none','②focal','③KD','④focal+KD']))
print('-'*68)
for ci,c in enumerate(CLASSES):
    row=f'{c:<12}'
    for k in cells_2x2:
        row += f"{cells_2x2[k]['per_class_f1'][ci]:<14.3f}"
    print(row)

클래스         ①none         ②focal        ③KD           ④focal+KD     
--------------------------------------------------------------------
Center      0.914         0.913         0.928         0.929         
Donut       0.823         0.792         0.838         0.815         
Edge-Loc    0.795         0.792         0.821         0.806         
Edge-Ring   0.974         0.973         0.980         0.979         
Loc         0.716         0.712         0.742         0.734         
Near-full   0.933         0.951         0.912         0.933         
Random      0.879         0.866         0.906         0.891         
Scratch     0.724         0.725         0.743         0.707         
none        0.989         0.988         0.989         0.989         


---
### Step 4 완료 = 모든 실험 종료
- 2×2 시너지 결과로 "불균형 처리 + KD"의 상호작용을 확인했습니다.
- 다음은 결과 그래프 생성 + 보고서 작성 + LLM 활용 로그 정리입니다.